# FIFA World Cup Predictor

This notebook demonstrates the complete pipeline:
1. Load data (World Cup matches + international results)
2. Engineer features (ELO ratings, rolling stats, head-to-head)
3. Train XGBoost classifier
4. Predict match outcomes
5. Simulate 2026 World Cup tournament
6. Generate reports and visualizations

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Reload modules to pick up code changes
import importlib
import sys
if 'predictor' in sys.modules:
    import predictor.config
    import predictor.data_loader
    import predictor.feature_engineer
    import predictor.model_trainer
    import predictor.predictor_api
    import predictor.simulator
    import predictor.reporter
    importlib.reload(predictor.config)
    importlib.reload(predictor.data_loader)
    importlib.reload(predictor.feature_engineer)
    importlib.reload(predictor.model_trainer)
    importlib.reload(predictor.predictor_api)
    importlib.reload(predictor.simulator)
    importlib.reload(predictor.reporter)

from predictor.config import DEFAULT_MODEL_PATH, DEFAULT_OUTPUT_DIR, RANDOM_SEED, FEATURE_COLS, OUTCOME_INT_MAP
from predictor.data_loader import DataLoader
from predictor.feature_engineer import FeatureEngineer
from predictor.model_trainer import ModelTrainer
from predictor.predictor_api import PredictorAPI
from predictor.simulator import TournamentSimulator
from predictor.reporter import ReportGenerator

DATA_DIR = '.'
CUTOFF_YEAR = 2018
N_RUNS = 1000
print('Imports OK')

Imports OK


## Step 1: Load Data

In [2]:
loader = DataLoader(DATA_DIR)
matches_df, players_df, cups_df, results_df = loader.load()
print(f'WC Matches:            {len(matches_df):>6} rows')
print(f'International results: {len(results_df):>6} rows  ({results_df["year"].min()}-{results_df["year"].max()})')
print(f'Players:               {len(players_df):>6} rows')
print(f'\nYear column info:')
print(f'  Type: {matches_df["Year"].dtype}')
print(f'  NaN count: {matches_df["Year"].isna().sum()}')
print(f'  Min: {matches_df["Year"].min()}, Max: {matches_df["Year"].max()}')
matches_df.head()

WorldCupMatches.csv: dropping 3720 row(s) with nulls in required columns.
WorldCupPlayers.csv: dropping 28715 row(s) with nulls in required columns.
results.csv: dropping 72 row(s) with nulls in required columns.


WC Matches:               964 rows
International results:  49215 rows  (1872-2026)
Players:                 9069 rows

Year column info:
  Type: int32
  NaN count: 0
  Min: 1930, Max: 2022


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930,1930-07-30 14:15:00,Final,Estadio Centenario,Montevideo,Uruguay,4.0,2.0,Argentina,,68346.0,1.0,2.0,LANGENUS Jean (BEL),SAUCEDO Ulises (BOL),CRISTOPHE Henry (BEL),405.0,1087.0,URU,ARG
2,1930,1930-07-27 14:45:00,Semi-finals,Estadio Centenario,Montevideo,Uruguay,6.0,1.0,Serbia,,79867.0,3.0,1.0,REGO Gilberto (BRA),SAUCEDO Ulises (BOL),BALWAY Thomas (FRA),202.0,1101.0,URU,YUG
3,1930,1930-07-26 14:45:00,Semi-finals,Estadio Centenario,Montevideo,Argentina,6.0,1.0,USA,,72886.0,1.0,0.0,LANGENUS Jean (BEL),VALLEJO Gaspar (MEX),WARNKEN Alberto (CHI),202.0,1088.0,ARG,USA
4,1930,1930-07-22 14:45:00,Group 1,Estadio Centenario,Montevideo,Argentina,3.0,1.0,Chile,,41459.0,2.0,1.0,LANGENUS Jean (BEL),CRISTOPHE Henry (BEL),SAUCEDO Ulises (BOL),201.0,1084.0,ARG,CHI


## Step 2: Feature Engineering

In [3]:
print('Building features (may take 1-2 minutes)...')
fe = FeatureEngineer(matches_df, players_df, results_df)
features_df = fe.build_features()
print(f'Feature matrix: {features_df.shape}')
print(f'Outcome distribution:')
print(features_df['outcome'].value_counts().to_string())
features_df.head()

Building features (may take 1-2 minutes)...
Feature matrix: (1928, 31)
Outcome distribution:
outcome
Home Win    750
Away Win    750
Draw        428


,focal_team,opponent,year,team_win_rate,team_draw_rate,team_loss_rate,team_avg_goals_scored,team_avg_goals_conceded,team_avg_goal_diff,opp_win_rate,...,h2h_avg_goal_diff,team_elo,opp_elo,elo_diff,stage_ordinal,tournament_weight,is_neutral,team_avg_goal_events_per_player,opp_avg_goal_events_per_player,outcome
0,France,Mexico,1930,0.310345,0.103448,0.586207,1.873563,3.149425,-1.275862,0.500000,...,NaN,1359.493100,1489.293758,-129.800657,1,1.0,1,0.0,0.0,Home Win
1,Mexico,France,1930,0.500000,0.125000,0.375000,2.125000,2.500000,-0.375000,0.310345,...,NaN,1489.293758,1359.493100,129.800657,1,1.0,1,0.0,0.0,Away Win
2,Uruguay,Argentina,1930,0.485294,0.191176,0.323529,1.786765,1.235294,0.551471,0.506757,...,-0.144444,1692.009862,1754.750405,-62.740543,7,1.0,1,0.0,0.0,Home Win
3,Argentina,Uruguay,1930,0.506757,0.216216,0.277027,1.939189,1.175676,0.763514,0.485294,...,0.144444,1754.750405,1692.009862,62.740543,7,1.0,1,0.0,0.0,Away Win
4,Uruguay,Serbia,1930,0.485294,0.191176,0.323529,1.786765,1.235294,0.551471,0.312500,...,7.000000,1682.230628,1500.788783,181.441845,5,1.0,1,0.0,0.0,Home Win


## Step 3: Train Model

In [4]:
trainer = ModelTrainer(features_df, cutoff_year=CUTOFF_YEAR, model_path=DEFAULT_MODEL_PATH)
result = trainer.train()
trainer.save()
print('Test set metrics:')
for k, v in result.metrics.items():
    print(f'  {k}: {v:.4f}')
print(f'\nTop 10 features by importance:')
print(result.feature_importance.nlargest(10).to_string())

Test set metrics:
  accuracy: 0.4688
  precision: 0.4125
  recall: 0.4688
  f1: 0.4344

Top 10 features by importance:
elo_diff                           0.066863
stage_ordinal                      0.057604
year                               0.044092
h2h_avg_goal_diff                  0.040012
h2h_draw_rate                      0.039742
team_avg_goals_conceded            0.039103
opp_avg_goal_events_per_player     0.038054
team_avg_goal_events_per_player    0.037634
opp_avg_goals_conceded             0.037400
opp_avg_goals_scored               0.037218


## Step 4: Test Predictions

In [5]:
api = PredictorAPI(result.model, fe)

test_pairs = [
    ('Brazil', 'Germany'),
    ('France', 'Argentina'),
    ('Spain', 'England'),
]
print('Sample match predictions:')
for home, away in test_pairs:
    try:
        pred = api.predict(home, away)
        print(f'{home:15} vs {away:15}  |  Home: {pred.home_win_prob:.3f}  Draw: {pred.draw_prob:.3f}  Away: {pred.away_win_prob:.3f}  =>  {pred.predicted_label}')
    except ValueError as e:
        print(f'  Skipping: {e}')

Sample match predictions:
Brazil          vs Germany          |  Home: 0.388  Draw: 0.034  Away: 0.577  =>  Away Win
France          vs Argentina        |  Home: 0.250  Draw: 0.344  Away: 0.406  =>  Away Win
Spain           vs England          |  Home: 0.466  Draw: 0.064  Away: 0.470  =>  Away Win


## Step 5: Setup 2026 World Cup Groups

In [6]:
# Official 2026 FIFA World Cup groups (draw held December 5, 2025)
# 48-team tournament across 12 groups
WC_2026_GROUPS = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['USA', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Curacao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Norway', 'Iraq'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

known = set(api._team_lookup.keys())
valid_groups = {}
for g, teams in WC_2026_GROUPS.items():
    valid = [t for t in teams if t.lower() in known]
    if len(valid) >= 2:
        valid_groups[g] = valid

valid_teams = [t for teams in valid_groups.values() for t in teams]
print(f'Valid groups: {len(valid_groups)}, teams: {len(valid_teams)}')
for g, t in valid_groups.items():
    print(f'  Group {g}: {t}')

Valid groups: 12, teams: 47
  Group A: ['Mexico', 'South Africa', 'South Korea', 'Czech Republic']
  Group B: ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland']
  Group C: ['Brazil', 'Morocco', 'Haiti', 'Scotland']
  Group D: ['USA', 'Paraguay', 'Australia', 'Turkey']
  Group E: ['Germany', 'Ivory Coast', 'Ecuador']
  Group F: ['Netherlands', 'Japan', 'Sweden', 'Tunisia']
  Group G: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
  Group H: ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay']
  Group I: ['France', 'Senegal', 'Norway', 'Iraq']
  Group J: ['Argentina', 'Algeria', 'Austria', 'Jordan']
  Group K: ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia']
  Group L: ['England', 'Croatia', 'Ghana', 'Panama']


## Step 6: Simulate Tournament & Generate Reports

In [7]:
if len(valid_groups) >= 12:
    simulator = TournamentSimulator(api, n_runs=N_RUNS, seed=RANDOM_SEED)
    sim_result = simulator.simulate(valid_teams, valid_groups)

    reporter = ReportGenerator(DEFAULT_OUTPUT_DIR)
    reporter.bar_chart(sim_result)
    reporter.feature_importance(result.feature_importance)
    reporter.summary_csv(sim_result)

    # Confusion matrix on test set
    test_df = features_df[features_df['year'] >= CUTOFF_YEAR]
    y_true = test_df['outcome'].tolist()
    y_pred_int = result.model.predict(test_df[list(FEATURE_COLS)])
    y_pred = [OUTCOME_INT_MAP[i] for i in y_pred_int]
    reporter.confusion_matrix(y_true, y_pred)

    print('\nTop 10 predicted 2026 WC winners:')
    for i, team in enumerate(sim_result.ranked_teams[:10], 1):
        prob = sim_result.win_probabilities[team]
        sf_prob = sim_result.semifinal_probabilities[team]
        print(f'  {i:2d}. {team:<20} win={prob:.3f}  semi={sf_prob:.3f}')
else:
    print(f'Only {len(valid_groups)} valid groups — need 12. Check team name mapping.')
    print('Groups missing teams from training data:')
    for g, teams in WC_2026_GROUPS.items():
        missing = [t for t in teams if t.lower() not in known]
        if missing:
            print(f'  Group {g}: {missing}')


Top 10 predicted 2026 WC winners:
   1. Netherlands          win=0.182  semi=0.371
   2. Argentina            win=0.180  semi=0.458
   3. Germany              win=0.171  semi=0.407
   4. France               win=0.102  semi=0.279
   5. Brazil               win=0.082  semi=0.306
   6. Norway               win=0.045  semi=0.180
   7. Switzerland          win=0.044  semi=0.186
   8. Colombia             win=0.043  semi=0.183
   9. Turkey               win=0.032  semi=0.286
  10. Spain                win=0.032  semi=0.216


Data Loading - Pulls in 964 World Cup matches (1930-2022), 49K international results, and 9K player records

Feature Engineering - Builds 31 features including ELO ratings, rolling stats (win/draw/loss rates, goals), head-to-head history, and player performance metrics. Creates 1,928 training examples.

Model Training - XGBoost classifier trained on pre-2018 data, tested on 2018+. Gets ~47% accuracy with ELO difference and stage being the most important features.

Prediction Testing - Shows sample predictions for matchups like Brazil vs Germany, with probabilities for home win/draw/away win

2026 World Cup Setup - Uses the official 48-team, 12-group format from the December 2025 draw

Tournament Simulation - Runs 1,000 Monte Carlo simulations and generates visualizations. Top predicted winners are Netherlands (18.2%), Argentina (18.0%), and Germany (17.1%)

Multi-class classification - You have 3 outcomes (Home Win, Draw, Away Win), and XGBoost handles multi-class problems well with its softmax objective

Handles imbalanced classes - Draws are less common (428 vs 750 wins each). The code uses sample weights with compute_sample_weight("balanced") which XGBoost supports natively

Feature importance - XGBoost provides built-in feature importance scores, which you're using to understand what drives predictions (ELO diff, stage, etc.)

Missing data tolerance - While you're using an imputer, XGBoost can handle sparse data and missing values internally, which is common in sports data

Non-linear relationships - Football outcomes depend on complex interactions between features (ELO + home advantage + tournament stage). Tree-based models like XGBoost capture these better than linear models

Hyperparameter tuning - The code does RandomizedSearchCV over 40 iterations tuning 7 parameters (depth, learning rate, subsample, etc.), and XGBoost is fast enough to make this practical

Proven track record - XGBoost is the go-to for tabular data competitions and sports prediction tasks

change training dates to more recent, look at group weights as a feature

Try NN and see if our accuracy changes in any way, change loss function or any other parameters to determine if its better 